In [8]:
import pandas as pd
import json

# 1. Đọc file JSON bằng thư viện json tiêu chuẩn
with open('dataset/cluster_summary.json', 'r', encoding='utf-8') as f:
    json_data = json.load(f)
with open('dataset/services.json', 'r', encoding='utf-8') as f:
    service_json = json.load(f)
# 2. Chuyển đổi dữ liệu JSON thành DataFrame
df_clusters = pd.DataFrame(json_data)

df_service_json = pd.json_normalize(service_json)

print(df_clusters.columns)

print(df_clusters.head())
df_jsonl = pd.read_json('dataset/alerts_sample.jsonl', lines=True)
# print(df_json.head())

# print(df_jsonl.head())

Index(['input_alerts', 'output_clusters', 'reduction_ratio', 'clusters'], dtype='object')
   input_alerts  output_clusters  reduction_ratio  \
0            20                2              0.9   
1            20                2              0.9   

                                            clusters  
0  {'cluster_id': 'c-000-000', 'alert_count': 19,...  
1  {'cluster_id': 'c-001-000', 'alert_count': 1, ...  


Build graph

In [11]:
import networkx as nx
G = nx.DiGraph()
for edge in service_json['edges']:
    # Thêm cạnh nối giữa 2 service, kèm theo thuộc tính loại kết nối (http, kafka,...)
    G.add_edge(edge['from'], edge['to'], type=edge['type'])
print(f"📌 TỔNG SỐ SERVICES ({G.number_of_nodes()}):")
# G.nodes trả về danh sách các node, chúng ta sắp xếp lại cho dễ nhìn
for node in sorted(G.nodes):
    print(f"  - {node}")

print("\n" + "="*50 + "\n")
print(f"🔌 TỔNG SỐ CONNECTIONS ({G.number_of_edges()}):")
for u, v, d in G.edges(data=True):
    connection_type = d.get('type', 'unknown')
    print(f"  [{u}] ----({connection_type})----> [{v}]")

📌 TỔNG SỐ SERVICES (14):
  - auth-svc
  - cart-redis
  - cart-svc
  - catalog-db
  - catalog-svc
  - checkout-svc
  - edge-lb
  - inventory-svc
  - kafka-events
  - notification-svc
  - payment-svc
  - payments-db
  - recommender-svc
  - search-svc


🔌 TỔNG SỐ CONNECTIONS (17):
  [edge-lb] ----(http)----> [auth-svc]
  [edge-lb] ----(http)----> [catalog-svc]
  [edge-lb] ----(http)----> [search-svc]
  [edge-lb] ----(http)----> [checkout-svc]
  [catalog-svc] ----(postgres)----> [catalog-db]
  [catalog-svc] ----(http)----> [recommender-svc]
  [search-svc] ----(postgres)----> [catalog-db]
  [checkout-svc] ----(http)----> [cart-svc]
  [checkout-svc] ----(http)----> [payment-svc]
  [checkout-svc] ----(http)----> [inventory-svc]
  [checkout-svc] ----(kafka)----> [notification-svc]
  [cart-svc] ----(redis)----> [cart-redis]
  [cart-svc] ----(http)----> [catalog-svc]
  [payment-svc] ----(postgres)----> [payments-db]
  [inventory-svc] ----(postgres)----> [catalog-db]
  [notification-svc] ----(kaf

In [ ]:
# --- 1. ĐỊNH NGHĨA GRAPH SCORER (REVERSE PAGERANK) ---
def run_graph_scorer(global_graph, target_services):
    """
    Trích xuất subgraph từ các service trong cluster, đảo ngược chiều cạnh 
    và tính điểm PageRank để tìm điểm gốc (Root Cause).
    """
    # Lấy đồ thị con chỉ gồm các service có trong cluster
    subgraph = global_graph.subgraph(target_services).copy()
    
    if len(subgraph) == 0:
        return {node: 0.0 for node in target_services}
    
    # ĐẢO NGƯỢC ĐỒ THỊ: Lỗi lan truyền từ Downstream -> Upstream, 
    # đảo ngược giúp nút gốc nhận được nhiều "phiếu bầu" PageRank nhất.
    reversed_subgraph = subgraph.reverse()
    
    try:
        # Tính PageRank trên đồ thị đảo ngược
        pagerank_scores = nx.pagerank(reversed_subgraph, alpha=0.85)
    except:
        # Phòng trường hợp đồ thị không hội tụ, dùng Degree Centrality làm phương án dự phòng
        pagerank_scores = nx.in_degree_centrality(reversed_subgraph)
        
    # Đảm bảo tất cả các service trong danh sách đều có điểm (kể cả node cô lập)
    return {node: pagerank_scores.get(node, 1.0 / len(target_services)) for node in target_services}

# --- 2. ĐỊNH NGHĨA TEMPORAL / SEVERITY SCORER ---
def run_temporal_scorer(fingerprints, target_services):
    """
    Tính điểm dựa trên số lượng fingerprints và mức độ nghiêm trọng (crit = 2, warn = 1)
    """
    severity_scores = {node: 0 for node in target_services}
    
    for fp in fingerprints:
        parts = fp.split('|')
        if len(parts) >= 3:
            svc, metric, severity = parts[0], parts[1], parts[2]
            if svc in severity_scores:
                # Gán trọng số: lỗi critical nặng hơn lỗi warning
                weight = 2.0 if severity.lower() == 'crit' else 1.0
                severity_scores[svc] += weight
                
    # Chuẩn hóa điểm về khoảng [0, 1] để cộng với Graph Score
    max_score = max(severity_scores.values()) if severity_scores.values() else 1
    if max_score == 0: max_score = 1
    
    normalized_scores = {svc: score / max_score for svc, score in severity_scores.items()}
    return normalized_scores
# --- 3. TIẾN HÀNH CHẠY CHO TỪNG CLUSTER ---
def extract_top_k_candidates(cluster_json, global_graph, K=3):
    """
    alpha: Trọng số kết hợp giữa Graph (alpha) và Temporal (1 - alpha)
    """
    final_results = {}
    
    for cluster in cluster_json['clusters']:
        cluster_id = cluster['cluster_id']
        services = cluster['services']
        fingerprints = cluster['fingerprints']
        
        # Chạy 2 bộ chấm điểm
        graph_scores = run_graph_scorer(global_graph, services)
        temporal_scores = run_temporal_scorer(fingerprints, services)
        
        # Kết hợp điểm (Hybrid Score)
        combined_data = []
        for svc in services:
            g_s = graph_scores.get(svc, 0.0)
            t_s = temporal_scores.get(svc, 0.0)
            hybrid_score = ((0.6 * g_s) + (0.4 * t_s))
            
            combined_data.append({
                'service': svc,
                'graph_score': round(g_s, 4),
                'temporal_score': round(t_s, 4),
                'hybrid_score': round(hybrid_score, 4)
            })
            
        # Chuyển thành DataFrame để sắp xếp và lọc Top-K
        df_candidates = pd.DataFrame(combined_data)
        top_k = df_candidates.sort_values(by='hybrid_score', ascending=False).head(K)
        
        final_results[cluster_id] = top_k.to_dict(orient='records')
        
    return final_results
top_candidates = extract_top_k_candidates(json_data, G, K=3)
print("=================== 🏆 TOP-K ROOT CAUSE CANDIDATES PER CLUSTER ===================")
for c_id, candidates in top_candidates.items():
    print(f"\n📦 Cluster ID: {c_id}")
    print("-" * 75)
    for index, cand in enumerate(candidates, 1):
        print(f"  Top {index}: {cand['service'].upper():<20} | Tổng điểm: {cand['hybrid_score']:.4f} (Graph: {cand['graph_score']:.4f}, Alert: {cand['temporal_score']:.4f})")

=================== 🏆 TOP-K ROOT CAUSE CANDIDATES PER CLUSTER ===================

📦 Cluster ID: c-000-000
---------------------------------------------------------------------------
  Top 1: CHECKOUT-SVC         | Tổng điểm: 0.5087 (Graph: 0.2646, Alert: 0.8750)
  Top 2: EDGE-LB              | Tổng điểm: 0.4677 (Graph: 0.3628, Alert: 0.6250)
  Top 3: PAYMENT-SVC          | Tổng điểm: 0.4447 (Graph: 0.0745, Alert: 1.0000)

📦 Cluster ID: c-001-000
---------------------------------------------------------------------------
  Top 1: PAYMENT-SVC          | Tổng điểm: 1.0000 (Graph: 1.0000, Alert: 1.0000)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
with open("dataset/incidents_history.json", 'r', encoding='utf-8') as f:
    data_json = json.load(f)
    # Lấy danh sách các incident bên trong mảng "incidents"
    history_incidents = data_json['incidents']

raw_fingerprints = []

# Duyệt qua từng cluster để bốc tách fingerprints
for cluster in json_data['clusters']:
    raw_fingerprints.extend(cluster['fingerprints'])

# Gộp lại thành 1 chuỗi lớn, thay dấu '|' thành dấu cách ' ' để phục vụ TF-IDF tốt hơn
current_incident_query = " ".join(raw_fingerprints).replace("|", " ")

print("=== CHUỖI FINGERPRINTS SAU KHI GOM ===")
print(current_incident_query)

# --- BƯỚC 3: TÍNH TOÁN ĐỘ TƯƠNG ĐỒNG TỪ KHÓA (TF-IDF + COSINE SIMILARITY) ---
# Trích xuất trường 'summary' để làm tập dữ liệu đối sánh văn bản
corpus = [inc['summary'] for inc in history_incidents]

# Khởi tạo bộ chuyển đổi từ ngữ sang Vector (loại bỏ các từ nối tiếng Anh phổ biến)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# Biến đổi dữ liệu sự cố hiện tại sang dạng Vector tương ứng
query_vector = vectorizer.transform([current_incident_query])

# Tính toán ma trận độ tương đồng Cosine giữa Query và 30 sự cố quá khứ
similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

# Gán điểm số tương đồng ngược lại vào danh sách incident gốc
for idx, score in enumerate(similarity_scores):
    history_incidents[idx]['similarity_score'] = round(float(score), 4)

# --- BƯỚC 4: TRÍCH XUẤT TOP-3 SỰ CỐ TƯƠNG ĐỒNG NHẤT ---
df_history = pd.DataFrame(history_incidents)
top_3_similar = df_history.sort_values(by='similarity_score', ascending=False).head(3)

# --- BƯỚC 5: IN KẾT QUẢ ĐẦU RA CHI TIẾT ---
print("="*24 + "TOP-3 SIMILAR HISTORICAL INCIDENTS RETRIEVED " + "="*24)
for rank, row in enumerate(top_3_similar.itertuples(), 1):
    print(f"\nTOP {rank}: ID [{row.id}] - Độ tương đồng từ khóa: {row.similarity_score * 100:.2f}%")
    print(f" Thời gian: {row.ts} | Mức độ: {row.severity.upper()}")
    print(f" Dịch vụ liên quan: {row.services_involved}")
    print(f" Nguyên nhân gốc: {row.root_cause_service.upper()} ({row.root_cause_class})")
    print(f" Tóm tắt quá khứ : {row.summary}")
    print(f" Cách xử lý (Remediation): {row.remediation}")
    print("-" * 95)


=== CHUỖI FINGERPRINTS SAU KHI GOM ===
cart-svc latency_p99_ms warn checkout-svc downstream_payment_error_rate crit checkout-svc latency_p99_ms crit checkout-svc latency_p99_ms warn checkout-svc request_drop_rate crit edge-lb p99_latency_ms crit edge-lb upstream_5xx_rate crit edge-lb upstream_5xx_rate warn notification-svc queue_depth crit notification-svc queue_lag_ms warn payment-svc db_connection_pool_used_ratio crit payment-svc db_connection_pool_used_ratio warn payment-svc error_rate crit payment-svc error_rate warn payment-svc latency_p99_ms crit recommender-svc cpu_utilization warn search-svc catalog_db_query_time_ms warn payment-svc latency_p99_ms crit
========================TOP-3 SIMILAR HISTORICAL INCIDENTS RETRIEVED ========================

TOP 1: ID [INC-2025-10-15] - Độ tương đồng từ khóa: 30.02%
 Thời gian: 2025-10-15T08:55:00Z | Mức độ: HIGH
 Dịch vụ liên quan: ['checkout-svc', 'inventory-svc']
 Nguyên nhân gốc: INVENTORY-SVC (infinite_retry)
 Tóm tắt quá khứ : Checkou

In [24]:
print("\n" + "Do có 2 cluster nên mỗi cluster sẽ có 1 kết quả phân loại riêng")

for cluster in json_data['clusters']:
    cluster_id = cluster['cluster_id']
    
    # 1. Gom toàn bộ fingerprints của cluster này thành 1 chuỗi text sạch
    cluster_query = " ".join(cluster['fingerprints']).replace("|", " ")
    
    # 2. Chuyển đổi sang vector và tính Cosine Similarity với kho lịch sử
    query_vector = vectorizer.transform([cluster_query])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # 3. Tìm vị trí Index của incident có điểm cao nhất (Top-1 / Nearest Neighbor)
    top_1_index = similarity_scores.argmax()
    max_score = similarity_scores[top_1_index]
    
    # 4. Trích xuất thông tin từ "Người hàng xóm gần nhất"
    matched_incident = history_incidents[top_1_index]
    
    # LẤY CLASS + ACTIONS 
    predicted_class = matched_incident['root_cause_class']
    recommended_actions = matched_incident['remediation']
    root_cause_svc = matched_incident['root_cause_service']
    matched_id = matched_incident['id']
    # 5. IN KẾT QUẢ PHÂN LOẠI TỰ ĐỘNG
    print(f"\nCluster ID: {cluster_id}")
    print(f"Top-1 Matched Incident: [{matched_id}] (Độ khớp: {max_score * 100:.2f}%)")
    print(f"Predicted Class       : {predicted_class.upper()}")
    print(f"Suspected Service     : {root_cause_svc.upper()}")
    print(f"Recommended Actions   : {recommended_actions}")
    print("-" * 85)


Do có 2 cluster nên mỗi cluster sẽ có 1 kết quả phân loại riêng

Cluster ID: c-000-000
Top-1 Matched Incident: [INC-2025-10-15] (Độ khớp: 30.30%)
Predicted Class       : INFINITE_RETRY
Suspected Service     : INVENTORY-SVC
Recommended Actions   : Add circuit breaker (Hystrix-style) 50% error → open. Cap retry = 3.
-------------------------------------------------------------------------------------

Cluster ID: c-001-000
Top-1 Matched Incident: [INC-2025-11-08] (Độ khớp: 27.17%)
Predicted Class       : CONNECTION_POOL_EXHAUSTION
Suspected Service     : PAYMENT-SVC
Recommended Actions   : Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.
-------------------------------------------------------------------------------------


In [27]:
import json
import numpy as np

# --- BƯỚC 1: KHỞI TẠO ĐỐI TƯỢNG CHỨA OUTPUT THEO SCHEMA ---
final_output = {
    "clusters_analyzed": len(json_data['clusters']),
    "results": []
}

# --- BƯỚC 2: RÁP NỐI VÀ ĐÓNG GÓI DỮ LIỆU ---
for cluster in json_data['clusters']:
    cluster_id = cluster['cluster_id']
    services = cluster['services']
    fingerprints = cluster['fingerprints']
    
    # 1. Thu thập kết quả từ Mảng 1 & Mảng 3 (Graph Scorer Output)
    # Lấy thông tin từ biến `top_candidates` mà bạn đã chạy ở trên
    cluster_top_graph = top_candidates.get(cluster_id, [])
    
    # Định dạng lại thành mảng lồng nhau [[service, score], ...] cho graph_top3
    graph_top3 = []
    for cand in cluster_top_graph[:3]:
        graph_top3.append([cand['service'], round(cand['hybrid_score'], 2)])
        
    # Xác định root_cause hàng đầu từ đồ thị hybrid
    predicted_root_cause = graph_top3[0][0] if graph_top3 else "unknown-svc"
    
    # 2. Thu thập kết quả từ Mảng 2 (RAG & Classifier 1-NN)
    cluster_query = " ".join(fingerprints).replace("|", " ")
    query_vector = vectorizer.transform([cluster_query])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    # Sắp xếp tìm các index khớp từ cao xuống thấp
    matched_indices = np.where(similarity_scores > 0)[0]
    sorted_matched_indices = matched_indices[np.argsort(similarity_scores[matched_indices])[::-1]]
    
    # 3. Áp dụng Logic đóng gói + Fallback nếu RAG rỗng
    if len(sorted_matched_indices) == 0:
        # Kịch bản Fallback (Không có sự cố lịch sử nào khớp)
        similar_incidents = []
        predicted_class = "UNKNOWN_HEURISTIC_MAPPED"
        confidence = round(float(graph_top3[0][1] * 0.5), 2) if graph_top3 else 0.30
        actions = [
            f"Collect diagnostics and profile resource allocation for {predicted_root_cause}.",
            "Inspect downstream system network latencies and database pool saturation profiles."
        ]
        method = "graph_only_fallback"
        reasoning = (
            f"No matching signatures found in historical incident catalogs. Root cause pinpointed to "
            f"[{predicted_root_cause}] based purely on dependency graph topology metrics."
        )
    else:
        # Kịch bản thông thường: Đọc từ Top-1 Match lịch sử
        top_1_idx = sorted_matched_indices[0]
        top_1_incident = history_incidents[top_1_idx]
        
        # Lấy tối đa 2 ID sự cố tương đồng nhất
        similar_incidents = [history_incidents[idx]['id'] for idx in sorted_matched_indices[:2]]
        predicted_class = top_1_incident['root_cause_class']
        
        # Tính điểm confidence phối hợp: 60% từ điểm đồ thị hybrid, 40% từ điểm cosine text match
        graph_weight = graph_top3[0][1] if graph_top3 else 0.5
        text_weight = similarity_scores[top_1_idx]
        confidence = round(float((0.6 * graph_weight) + (0.4 * text_weight)), 2)
        
        # Băm nhỏ chuỗi remediation văn bản thành mảng các chuỗi hành động sạch
        actions = [act.strip() for act in top_1_incident['remediation'].split('. ') if act.strip()]
        method = "graph+rag_knn"
        reasoning = (
            f"Root cause suspected at [{predicted_root_cause}] via system dependency tracking. "
            f"Cluster symptoms highly correlate with historical outage [{top_1_incident['id']}] "
            f"classified under root cause class '{predicted_class}'."
        )
        
    # 4. Đóng gói thành phần cấu trúc JSON đúng định dạng yêu cầu
    cluster_schema_payload = {
        "cluster_id": cluster_id,
        "graph_top3": graph_top3,
        "root_cause": predicted_root_cause,
        "class": predicted_class,
        "confidence": confidence,
        "actions": actions,
        "reasoning": reasoning,
        "similar_incidents": similar_incidents,
        "method": method
    }
    
    final_output["results"].append(cluster_schema_payload)
with open('results/rca_output.json', 'w', encoding='utf-8') as json_file:
    json.dump(final_output, json_file, indent=2, ensure_ascii=False)